# Week 3 · Day 6: Evaluation and regularization

**Course:** Labs 07–09: Evaluation, overfitting, regularization

The teaching question is **“Is this source's absolute G magnitude below 4?”** Class 1 means yes. This is a brightness label, not a verified stellar type. Only `bp_rp` (color) enters the model; magnitude and parallax are used to construct the answer and must stay out of the inputs.

Day 1 saves a source-disjoint 60/20/20 split. Days 2–6 explore training and validation; Day 7 evaluates the reserved test set. Absolute magnitudes are uncorrected for dust, and the archive sample may not represent all Gaia sources.

## Setup

Run this setup cell first. It finds the Gaia Explorer folder, imports NumPy and pandas, and loads the shared data and plotting helpers. `ROOT` is the project folder; `OUTPUT` is where the Week 3 CSV files are saved. The [prerequisites notebook](week3_prerequisites.ipynb) explains the import and kernel steps in detail.

In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
from IPython.display import display

# Jupyter may start in the project folder or in its notebooks folder.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src/classification.py').is_file():
    raise RuntimeError('Open Jupyter from gaia-explorer or gaia-explorer/notebooks.')
sys.path.insert(0, str(ROOT))
from src import classification as data, visualize
# Refresh the data loader if this kernel imported an older version.
data = importlib.reload(data)
visualize = importlib.reload(visualize)  # Pick up updated plot labels and layout.

DATA_PATH = None  # None uses the shared W2 cache; a missing cache is downloaded.
OUTPUT = ROOT / 'data/processed/week3'
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)


## Load the Day 1 data

Day 1 saved `learning_data.csv` with a `split` column. This cell reads it and selects the training, validation, and reserved test rows. `source_id` is read as text so its long digits stay intact.

`X_train` and `X_val` contain only `bp_rp`, the color input. `y_train` and `y_val` contain `is_bright`, the known 0/1 answers. The code prepares `test` but does not use it to choose a model; Day 7 will evaluate it.

In [ ]:
prepared_path = OUTPUT / 'learning_data.csv'
if not prepared_path.is_file():
    raise FileNotFoundError('Run Week 3 Day 1 first to create learning_data.csv.')

# Keep source IDs as text so their digits are not rounded.
gaia = pd.read_csv(prepared_path, dtype={'source_id': 'string'})
train = gaia.loc[gaia['split'] == 'train'].copy()
validation = gaia.loc[gaia['split'] == 'validation'].copy()
test = gaia.loc[gaia['split'] == 'test'].copy()
feature_columns = ['bp_rp']
X_train = train[feature_columns]
y_train = train['is_bright']
X_val = validation[feature_columns]
y_val = validation['is_bright']
print('Training rows:', len(train), '| Validation rows:', len(validation))


## 1. Prepare model choices

We try **nine combinations**: polynomial degrees `1`, `3`, and `6`, each with `C` equal to `0.01`, `1.0`, or `100.0`. Because `bp_rp` is our only input, degree 3 creates color, color², and color³; degree 6 adds powers up to color⁶. Higher degrees let the probability curve bend more. `include_bias=False` avoids adding a constant input because logistic regression already learns an intercept.

`C` controls how strongly training restrains large weights: `0.01` is stronger restraint and `100.0` is weaker. For each combination, `make_pipeline` creates polynomial features, scales them, and fits logistic regression **using training rows only**. `max_iter` caps training iterations, while `tol` controls when the solver considers itself close enough to stop. The pipeline uses the training scaling when it later sees validation rows.

`predict_proba(... )[:, 1]` gets each star's estimated probability of **class 1** (`abs_g_mag < 4`). `train_p` is for training stars; `val_p` is for validation stars. `log_loss` averages how well these probabilities match the true labels, with smaller loss being better. Its reported value does not include the training regularization penalty.

`rows` collects the degree, `C`, and two losses for the table. `models` stores the fitted models in the **same order** as their table rows. `display(comparison)` shows all nine combinations. A model may fit training rows closely yet do worse on validation rows; comparing both losses helps us notice that.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

rows = []
models = []
for degree in [1, 3, 6]:
    for C in [0.01, 1.0, 100.0]:
        model = make_pipeline(
            PolynomialFeatures(degree=degree, include_bias=False),
            StandardScaler(),
            LogisticRegression(C=C, max_iter=5000, tol=1e-9),
        )
        model.fit(X_train, y_train)
        train_p = model.predict_proba(X_train)[:, 1]
        val_p = model.predict_proba(X_val)[:, 1]
        rows.append({'degree': degree, 'C': C,
                     'train_log_loss': log_loss(y_train, train_p),
                     'validation_log_loss': log_loss(y_val, val_p)})
        models.append(model)
comparison = pd.DataFrame(rows)
display(comparison)

## 2. Pick using validation loss

`idxmin()` finds the table row with the **smallest validation log loss**. Because `models` and the table were filled together, `models[best_index]` retrieves the fitted model for that row. `choice` holds that row's degree, `C`, training loss, and validation loss; `to_frame('selected')` displays those values vertically.

The selected model predicts a class 1 probability for each validation star. At the fixed threshold `0.5`, a probability of `0.7` becomes label `1`, while `0.3` becomes label `0`. Changing the threshold would change these labels, not the fitted weights or probabilities. Day 3 explored other thresholds; we keep `0.5` here so the comparison is simple.

We use validation loss to **select** the model. We do not look at the test rows while making this choice; Day 7 uses them for the final check.

In [ ]:
best_index = comparison['validation_log_loss'].idxmin()
best_model = models[best_index]
choice = comparison.loc[best_index]
display(choice.to_frame('selected'))
probabilities = best_model.predict_proba(X_val)[:, 1]
threshold = 0.5  # Fixed decision rule; Day 3 explored alternatives.
predicted = (probabilities >= threshold).astype(int)

## 3. Calculate metrics directly

This cell compares `predicted` with `y_val`, the validation stars' true labels. **Accuracy** is the fraction of all stars classified correctly. **Precision** asks: among stars predicted as class 1, how many really are class 1? **Recall** asks: among all actual class 1 stars, how many did the model find? **F1** combines precision and recall. `zero_division=0` reports 0 when a score cannot be calculated, for example when no stars are predicted as class 1.

The **constant baseline** is a simple guess that ignores color. `y_train.mean()` is the fraction of training stars in class 1. If 20 of 100 training stars are class 1, it equals `0.20`. `np.full(len(y_val), 0.20)` would then give **every validation star probability 0.20**. Comparing the baseline's log loss with the selected model's log loss asks whether using color produces better validation probabilities than that constant guess. Lower log loss is better.

The first plot shows a **confusion matrix** (correct predictions and the two kinds of mistakes) and a **precision–recall curve**. The second figure separates actual 0/1 labels in its top panel from predicted probabilities in its bottom panel. The orange probability curve crosses a dashed `0.5` threshold. These are validation plots, not final test results.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print('Accuracy:', accuracy_score(y_val, predicted))
print('Precision:', precision_score(y_val, predicted, zero_division=0))
print('Recall:', recall_score(y_val, predicted, zero_division=0))
print('F1:', f1_score(y_val, predicted, zero_division=0))
baseline_p = np.full(len(y_val), y_train.mean())
print('Constant baseline log loss:', log_loss(y_val, baseline_p))
print('Selected model log loss:', log_loss(y_val, probabilities))
visualize.plot_binary_evaluation(y_val, probabilities, threshold=threshold)
visualize.plot_color_probability(best_model, validation, threshold=threshold)

## 4. Save the choice for Day 7

`settings` records the chosen polynomial degree, `C`, threshold, label cut, split seed, and input feature. `comparison` records all nine model results. The CSV files let Day 7 use the same choice without silently choosing a new model based on test results.

The `data_sha256` value is a fingerprint of Day 1's saved CSV. It lets Day 7 detect if that file changed after the model choice was made. If you rerun Day 1 with different data or labels, rerun Day 6 before Day 7 so the saved choice belongs to the current data.

In [ ]:
import hashlib
fingerprint = hashlib.sha256(prepared_path.read_bytes()).hexdigest()
settings = pd.DataFrame([{
    'degree': int(choice['degree']), 'C': choice['C'], 'threshold': threshold,
    'label_cut_mag': 4.0, 'split_seed': 42, 'features': 'bp_rp',
    'data_sha256': fingerprint,
}])
settings.to_csv(OUTPUT / 'experiment_settings.csv', index=False)
comparison.to_csv(OUTPUT / 'validation_model_comparison.csv', index=False)
print('Saved model choice for Day 7.')

## Reflection questions and answers

1. **Must the most flexible model win?** No. Degree 6 gives the model more ways to bend its probability curve than degree 1. That may help if the relationship is genuinely curved, but it may also fit chance patterns in the training stars. For example, degree 6 could have lower *training* loss yet higher *validation* loss than degree 3. The best choice here is the one with the lowest validation log loss in our comparison, not automatically the highest degree or largest `C`.

2. **Why compare validation loss?** Training loss describes stars the model already learned from; a very flexible model can look good there even if it performs poorly on new stars. Validation stars were kept out of fitting, so their log loss checks how well the predicted probabilities match new 0/1 labels. Lower validation log loss is better for this choice. We still reserve the separate test set for Day 7 because we used validation results to choose the model.

3. **Why save choices before testing?** Day 6 chooses the polynomial degree and `C` from validation loss and fixes the label threshold at 0.5. Saving those settings lets Day 7 use the same choices when it checks the reserved test stars. If we looked at test results and then changed degree, `C`, or threshold to improve them, the test set would influence the model and stop being a fair final check. If Day 1's data changes, rerun Day 6 so the saved choice matches it.


## References

Course labs: `../ML-coursework/W3/Optional Labs/` in the workspace.

[ESA Gaia DR3](https://www.cosmos.esa.int/web/gaia/dr3) · [Extinction guidance](https://www.cosmos.esa.int/web/gaia/edr3-extinction-law) · [Scikit-learn logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)